# One request, traced through the four planes

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 3 §3.1–3.4 · type: concept-lab

**The promise:** by the end you can take any user request, watch it flow through the four planes — **model · orchestration · data · infrastructure** — and name, at every hop, which of the four forces (**cost · latency · reliability · safety**) is being spent.

This is the recommended **first notebook to run** in the whole repo. It is fully offline and free: no API key, no services, nothing to install beyond the standard library. Everything is mocked by design so Part I stays zero-friction.

## 🧠 Why this matters

Strip any production agentic system down and you find the same parts arranged the same way: an **orchestration** loop drives an **agent**, which consults the **model** for decisions, calls **tools** to act, reads and writes **data** and memory, all running on **infrastructure** that provides compute, queues, and observability. The model is one organ in a much larger body (§3.1).

The trap that keeps engineers junior is believing *the model is the system* — that a smart enough model plus a clever enough prompt is the whole job. It is exactly backwards. The model supplies **intelligence**; your system supplies the **engineering** that makes it fast, cheap, reliable, and safe at scale (§3.2). Almost everything that determines those four properties lives in the *other three* planes.

This notebook makes that concrete. We build a deliberately tiny mock agent, instrument every hop, and read a labeled trace. Hold this picture in your head and every later chapter snaps onto it: you will be able to point at any topic and say *which plane* it lives in and *which force* it pushes on. That is the orientation the whole book hangs on.

## Objectives & prereqs

**By the end you can:**
- Trace a single request end-to-end and label each hop as **model**, **orchestration**, **data**, or **infrastructure**.
- Read a per-hop trace of latency and estimated cost, and say which hop dominates.
- Name which of the four forces — **cost, latency, reliability, safety** — a design change spends.
- Overlay the four planes onto the `capstone-project/` directory tree, so the repo's layout stops looking arbitrary.

**Prereqs:** Chapters 1–2 read, Chapter 3 read alongside. No prior notebook required — this is the entry point.

**Packages:** standard library only. No external dependencies, no API key, no network. (When a real system would call a model, we return a canned, realistic response instead.)

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import random
import time
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default) keeps everything offline, free, and deterministic. This notebook
# is mock-only by design: there is no live path to enable. The switch is here so the
# pattern is identical to every other notebook in the repo — you will flip it to 0
# in later chapters once a real model call earns its cost.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed everything stochastic so the trace is identical on every run and in CI.
SEED = 7
random.seed(SEED)

print(f"MOCK mode: {MOCK}  (offline, free, deterministic)")
print("No API key required. Nothing leaves this machine.")

## The four planes, in one table

The cleanest way to hold the architecture in your head is as four **planes**, each answering a different question (§3.3). Almost every component in the book belongs to exactly one of them.

| Plane | Answers the question… | Examples |
|---|---|---|
| **Model** | *How does it decide?* | the LLM(s), guardrails, evaluation |
| **Orchestration** | *What does it do next?* | the agent loop, planning/routing, tools |
| **Data** | *What does it know?* | retrieval (RAG), databases, memory |
| **Infrastructure** | *What does it run on?* | compute, queues/workers, deployment, observability |

The planes are a *thinking tool*, not a rulebook. Their value shows up when something breaks: a bad answer is usually a **data** problem (wrong context retrieved), a runaway loop is an **orchestration** problem (no termination condition), a timeout is an **infrastructure** problem, and a confidently-wrong fact reached for where retrieval was needed is a **model** problem. Naming the plane is half of solving it.

In [ ]:
# The four planes and the four forces, named exactly as the book names them (§3.3, §3.4).
PLANES = ("model", "orchestration", "data", "infrastructure")
FORCES = ("cost", "latency", "reliability", "safety")


@dataclass
class Hop:
    """One step a request takes through the system."""
    name: str            # human-readable step name
    plane: str           # which of the four planes this hop belongs to
    latency_ms: float    # how long this hop took
    cost_usd: float      # estimated marginal cost of this hop
    force: str           # the force this hop most obviously spends
    note: str = ""       # one-line "why this hop exists"

    def __post_init__(self):
        # Fail loudly if a hop is mislabeled — the whole point is correct labels.
        assert self.plane in PLANES, f"unknown plane: {self.plane}"
        assert self.force in FORCES, f"unknown force: {self.force}"


@dataclass
class Trace:
    """An ordered record of every hop a request took."""
    hops: list = field(default_factory=list)

    def add(self, hop: Hop) -> None:
        self.hops.append(hop)

    @property
    def total_ms(self) -> float:
        return sum(h.latency_ms for h in self.hops)

    @property
    def total_usd(self) -> float:
        return sum(h.cost_usd for h in self.hops)


print(f"planes: {PLANES}")
print(f"forces: {FORCES}")

## A deliberately tiny mock agent

Here is the smallest system that still has all four planes: one **retrieval** step (data), one **model** step (orchestration asks the model to decide), and one **tool** call (orchestration acts), all wrapped in an orchestration loop running on mock infrastructure.

Every component below is **fully mocked** — canned, realistic responses with realistic-looking latencies. No network, no key, no randomness beyond the seed we set. The shapes (a retriever returning documents, a model returning a decision, a tool returning a result) are the same ones you will build for real in Parts III–IV; here we only care about the *flow*.

In [ ]:
# --- DATA plane: a tiny in-memory "knowledge base" (3 docs, no external service). ---
DOCS = {
    "refund-policy": "Refunds are issued within 14 days of purchase for unused items.",
    "shipping": "Standard shipping takes 3-5 business days; express takes 1-2.",
    "warranty": "All devices carry a 1-year limited warranty against defects.",
}


def retrieve(query: str, trace: "Trace") -> str:
    """DATA plane: fetch the most relevant doc. Mocked with a trivial keyword match."""
    # Pick the doc whose key shares the most words with the query.
    q = set(query.lower().replace("?", "").split())
    best = max(DOCS, key=lambda k: len(q & set(k.split("-"))))
    doc = DOCS[best]
    trace.add(Hop(
        name=f"retrieve({best})", plane="data",
        latency_ms=18.0, cost_usd=0.00002, force="latency",
        note="vector lookup: the 'what does it know?' plane",
    ))
    return doc


# --- MODEL plane (called by orchestration): decide what to do, given the context. ---
def call_model(prompt: str, context: str, trace: "Trace") -> dict:
    """MODEL plane: return a decision. MOCK returns a canned, realistic response."""
    if MOCK:
        # Canned, realistic model output: it decides to call a tool first.
        decision = {
            "action": "lookup_order",
            "args": {"order_id": "A-1029"},
            "rationale": "User asked about a refund; need order status first.",
        }
    else:  # pragma: no cover - no live path in Part I, kept for shape parity
        raise RuntimeError("Part I is mock-only; no live model call is wired here.")
    # A real model call is the slow, expensive hop: hundreds of ms, real tokens.
    trace.add(Hop(
        name="model.decide", plane="model",
        latency_ms=720.0, cost_usd=0.00210, force="cost",
        note="the LLM: the 'how does it decide?' plane",
    ))
    return decision


# --- ORCHESTRATION plane: a tool the agent can call to act on the world. ---
def lookup_order(order_id: str, trace: "Trace") -> str:
    """ORCHESTRATION plane (tool): a mocked side-effecting action."""
    trace.add(Hop(
        name=f"tool.lookup_order({order_id})", plane="orchestration",
        latency_ms=45.0, cost_usd=0.00001, force="reliability",
        note="a tool call: the 'what does it do next?' plane acting on the world",
    ))
    return f"Order {order_id}: delivered 3 days ago, within the 14-day refund window."

## Wire it together — the orchestration loop

The orchestration plane is the conductor: it drives the loop, decides the order of hops, and enforces the termination condition. Below, a single `handle_request` runs the whole flow and records one `Hop` per step, plus the **infrastructure** hops (admission/observability) that wrap every real request but are easy to forget.

In [ ]:
def handle_request(user_msg: str) -> Trace:
    """ORCHESTRATION plane: drive one request through all four planes, tracing each hop."""
    trace = Trace()

    # INFRASTRUCTURE: every request first crosses the boundary — auth, queueing,
    # a span opened for observability. Cheap, but it is a real hop and a real plane.
    trace.add(Hop(
        name="ingress (auth + enqueue + span)", plane="infrastructure",
        latency_ms=12.0, cost_usd=0.0, force="reliability",
        note="the 'what does it run on?' plane: admission + observability",
    ))

    # DATA: fetch context for the request.
    context = retrieve(user_msg, trace)

    # MODEL: ask the model what to do next, given the context.
    decision = call_model(user_msg, context, trace)

    # ORCHESTRATION: execute the model's chosen tool (the loop's "act" step).
    if decision["action"] == "lookup_order":
        lookup_order(decision["args"]["order_id"], trace)

    # A second model pass would normally compose the final answer; we add it as one
    # "compose" hop to keep the trace short and the flow obvious.
    trace.add(Hop(
        name="model.compose_answer", plane="model",
        latency_ms=540.0, cost_usd=0.00160, force="cost",
        note="second model pass: turn the tool result into a grounded reply",
    ))

    # INFRASTRUCTURE: close the span / write the log line on the way out.
    trace.add(Hop(
        name="egress (log + close span)", plane="infrastructure",
        latency_ms=6.0, cost_usd=0.0, force="safety",
        note="observability: you cannot fix what you cannot see",
    ))

    return trace


trace = handle_request("How do refunds work for my order?")
print(f"request handled in {len(trace.hops)} hops")

## 🔮 Predict: which hop dominates latency?

You are about to print the trace. Before you run the next cell, **predict**: of all the hops — ingress, retrieve, the model decision, the tool call, the compose step, egress — which **single hop** eats the most wall-clock time? And which **plane** does it belong to?

Write your guess down. Then run the cell and read the trace.

In [ ]:
def print_trace(trace: Trace) -> None:
    """Print a labeled, per-hop trace: plane, latency, est. cost, and the force spent."""
    print(f"{'#':>2}  {'plane':<15} {'hop':<34} {'ms':>7} {'$':>9}  force")
    print("-" * 86)
    for i, h in enumerate(trace.hops, 1):
        print(f"{i:>2}  {h.plane:<15} {h.name:<34} {h.latency_ms:>7.1f} {h.cost_usd:>9.5f}  {h.force}")
    print("-" * 86)
    print(f"{'':>2}  {'TOTAL':<15} {'':<34} {trace.total_ms:>7.1f} {trace.total_usd:>9.5f}")

    # Roll up time by plane so the dominant plane is obvious.
    print("\nshare of latency by plane:")
    for plane in PLANES:
        ms = sum(h.latency_ms for h in trace.hops if h.plane == plane)
        share = ms / trace.total_ms if trace.total_ms else 0
        bar = "#" * round(share * 30)
        print(f"  {plane:<15} {ms:>7.1f} ms  {share:>5.0%}  {bar}")


print_trace(trace)

**What you just saw.** The two **model** hops dominate the clock — by a wide margin — even though they are only *two of six* hops. The retrieval, the tool call, and both infrastructure hops are nearly free by comparison. This is the lesson §3.1 is built around, made literal: the model is a small box in the diagram, but on the latency-and-cost axes it is the heavyweight, while everything that determines *correctness* lives in the cheaper planes around it.

Notice also the `force` column. The model hops spend **cost**; retrieval spends **latency**; the tool call is where **reliability** is at risk (an external action can fail); the egress log is a **safety**/observability concern. Every hop spends something.

## Toggle one knob — what a cache hit buys you

Now change exactly one thing and re-read the trace. Suppose the model's *decision* for this kind of request is cacheable (we have seen this question shape before), so the first model hop becomes a near-instant **cache hit** on the infrastructure plane instead of a fresh model call. Predict what happens to the totals before you run it.

In [ ]:
def handle_request_cached(user_msg: str) -> Trace:
    """Same flow, but the first model decision is served from cache (infra plane)."""
    trace = Trace()
    trace.add(Hop("ingress (auth + enqueue + span)", "infrastructure", 12.0, 0.0,
                  "reliability", "admission + observability"))
    retrieve(user_msg, trace)

    # The knob: instead of a 720ms / $0.0021 model call, a cache hit.
    trace.add(Hop("cache.get(decision)  HIT", "infrastructure", 3.0, 0.0,
                  "latency", "a cached decision: trade memory for time and cost"))

    lookup_order("A-1029", trace)
    trace.add(Hop("model.compose_answer", "model", 540.0, 0.00160, "cost",
                  "second model pass still runs"))
    trace.add(Hop("egress (log + close span)", "infrastructure", 6.0, 0.0, "safety",
                  "observability"))
    return trace


cached = handle_request_cached("How do refunds work for my order?")

print(f"{'metric':<16}{'no cache':>12}{'with cache':>14}{'delta':>12}")
print("-" * 54)
d_ms = cached.total_ms - trace.total_ms
d_usd = cached.total_usd - trace.total_usd
print(f"{'total latency':<16}{trace.total_ms:>10.1f}ms{cached.total_ms:>12.1f}ms{d_ms:>10.1f}ms")
print(f"{'total cost':<16}{trace.total_usd:>11.5f}{cached.total_usd:>13.5f}{d_usd:>12.5f}")

**What you just saw.** Swapping one model hop for a cache hit cut latency and cost in a single move — by replacing an expensive **model**-plane hop with a cheap **infrastructure**-plane one. You did not touch the model or the prompt. That is the whole game: the leverage was in a *different plane* than the one (the model) people instinctively reach for.

## 🎯 Senior lens

Every design choice later in this book moves one of the four forces — **cost, latency, reliability, safety** — and you can usually *see* the move in a trace like this one (§3.4). Caching traded memory for latency and cost. Adding a second model pass would buy answer quality at the price of both. Adding a guardrail or a human-approval step buys safety at the price of latency. A retry buys reliability at the price of latency and a little cost.

Juniors optimize one force and are surprised when another breaks ("I made it smarter!" — and ten times slower and five times costlier). Architects state the trade-off *out loud*: "we accept higher latency on this path to halve cost, and we cap autonomy here to bound the safety blast radius." When you can name which force a change spends *and what it buys*, you are reasoning like a senior. The trace is how you replace vibes with evidence — which is exactly what Part VI's observability tooling industrializes.

## ⚠️ Pitfall: mistaking "the model" for "the system"

The trace makes the most expensive misconception in this field concrete. When the answer is wrong, the instinct is to blame the model and reach for a bigger one. But look back at where the hops live: a wrong answer is most often a **data**-plane miss (we retrieved the wrong doc, so the model reasoned correctly over bad context), or an **orchestration** bug (the loop called the wrong tool, or never terminated), or an **infrastructure** failure (a tool timed out and we silently returned a stub).

Most failures live in the three planes that are *not* the model. Swapping in a smarter model raises the ceiling, but if the bug is in another plane, a better model just produces a more fluent wrong answer. **The fix:** when something breaks, locate the plane first, then fix it there. Reaching for a bigger model before you have named the plane is how teams burn budget without moving the needle.

## Recap

- Every agentic request flows through **four planes** — model, orchestration, data, infrastructure — and naming the plane a problem lives in is half of solving it.
- The **model** hops dominate latency and cost, yet they are a *small* part of the system; correctness mostly lives in the other three planes.
- Every hop spends one of **four forces** — cost, latency, reliability, safety — and you can read which in the trace.
- A design change (a cache, a retry, a guardrail, a second pass) is just *moving one force at the expense of another* — and a senior says which, out loud.
- Most failures are **not** the model. Locate the plane before you reach for a bigger model.

## Exercises

Each exercise *changes* something and asks you to predict the result before running.

1. **Which plane is Ch N about?** For five chapters you have not read yet — say tool use (Ch 12), RAG (Ch 13), memory (Ch 14), evaluation (Ch 18), and deployment (Ch 24) — write down which **plane** each one fills in. Then skim the table of contents and check. (Answer key: orchestration, data, data, model, infrastructure.)
2. **Add a retry.** Add a hop that retries the `lookup_order` tool once after a simulated failure. Predict the effect on total latency and on the **reliability** force, then add it to `handle_request` and re-read the trace.
3. **Second model pass.** Make `call_model` run twice (decide, then re-plan) before composing. Predict which plane's share of latency grows, then measure it.
4. **Name the force.** Pick any feature you have built or used (autocomplete, a fraud check, a chatbot). Trace it as 4–6 hops on paper, label each hop's plane, and name the dominant force. Which plane were you neglecting?

In [ ]:
# Exercise scratch space — your code here.

In [ ]:
# Exercise scratch space — your code here.

## Next

- **Overlay the planes on the capstone.** This trace is the mental map of the whole `capstone-project/` layout (Appendix C). As you read on, picture the capstone tree with each top-level area colored by plane:

  ```
  capstone-project/
    llm/                         -> MODEL           (the client, guardrails, evals)
    agent/                       -> ORCHESTRATION   (the loop, tools, routing)
    rag/  memory/                -> DATA            (retrieval, databases, memory)
    api/  infra/  observability/ -> INFRASTRUCTURE  (serving, queues, tracing)
  ```

- **Next notebook:** Part II begins the software-engineering foundation everything else stands on — start with its first chapter folder under [`../_parts/part-02-engineering-foundations.md`](../_parts/part-02-engineering-foundations.md).
- **Book:** keep §3.3 (the four planes) and §3.4 (the four forces) within reach — every later chapter is an expansion of one box in this picture.
- **Where this leads:** the tracing you faked here becomes real in [`blueprints/observability-stack/`](../../blueprints/observability-stack/), where spans, latency, and cost per hop are captured for an actual running agent.